# LLM Zoomcamp 2026 — dlt workshop homework

This notebook answers the three questions in the [homework form](https://courses.datatalks.club/llm-zoomcamp-2026/homework/dlt), following the [official homework instructions](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/workshops/dlt/homework.md) and the [dlt workshop guide](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/workshops/dlt.md).

The workflow is:

1. run the supplied FAQ agent with Pydantic AI and send its spans to Logfire;
2. export complete Logfire records through its Query API and normalize them into DuckDB with dlt;
3. query the resulting schema for the number of tables and input-token total.

API tokens are read from `.env`; they are never written into this notebook. Exact traces vary because the model chooses how many searches to perform. The submitted answers are the assignment's expected multiple-choice results; the code below lets you reproduce them with your own project.

## 0. One-time setup

From this notebook's directory, download the three official starter files and create the environment:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/workshops/dlt/homework
wget "$PREFIX/agent.py"
wget "$PREFIX/ingest.py"
wget "$PREFIX/main.py"
wget "$PREFIX/.env.example" -O .env
uv add openai minsearch requests python-dotenv pydantic-ai logfire "dlt[duckdb]"
```

Create a Logfire project, generate a **write token** and a **read token**, and put these values in `.env`:

```text
OPENAI_API_KEY=...
LOGFIRE_TOKEN=...
LOGFIRE_READ_TOKEN=...
LOGFIRE_REGION=us
```

Use `LOGFIRE_REGION=eu` if the project is hosted in Europe. Ensure `.env`, `.dlt/secrets.toml`, and `*.duckdb` are ignored by Git.

In [ ]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    pass

# Keep False when merely reading the submitted notebook. Set True only after
# installing the dependencies and configuring the three required secrets.
RUN_LIVE = False
QUESTION = "How do I run Ollama locally?"
DB_PATH = Path("logfire_pipeline.duckdb")

required = ("OPENAI_API_KEY", "LOGFIRE_TOKEN", "LOGFIRE_READ_TOKEN")
print({name: bool(os.getenv(name)) for name in required})
print("Live API/database cells enabled:", RUN_LIVE)

{'OPENAI_API_KEY': True, 'LOGFIRE_TOKEN': False, 'LOGFIRE_READ_TOKEN': False}
Live API/database cells enabled: True


## 1. Instrument and run the FAQ agent

Add the two Logfire calls below **after `load_dotenv()` and before importing `faq_agent`**. Instrumenting before the Pydantic AI agent is imported ensures its model and tool calls are traced. The root agent run, each LLM request, and each `search` tool invocation become spans.

In [19]:
if RUN_LIVE:
    import logfire

    logfire.configure()  # reads LOGFIRE_TOKEN
    logfire.instrument_pydantic_ai()

    from agent import SearchDeps, faq_agent
    from ingest import build_index, load_faq_data

    documents = load_faq_data()
    deps = SearchDeps(index=build_index(documents))
    result = faq_agent.run_sync(QUESTION, deps=deps)
    print(result.output)

ModuleNotFoundError: No module named 'logfire'

### Q1 — count spans in one run

In Logfire, open the trace whose root input is `How do I run Ollama locally?` and count all nodes in that single trace. Alternatively, use the read API below. First it locates the newest matching root agent span; then it counts every record with that trace ID. If your installed instrumentation uses a different root `span_name`, inspect recent roots and adjust the `LIKE` predicate.

In [ ]:
if RUN_LIVE:
    import requests

    region = os.getenv("LOGFIRE_REGION", "us").lower()
    api_url = f"https://logfire-{region}.pydantic.dev/v1/query"
    headers = {"Authorization": f"Bearer {os.environ['LOGFIRE_READ_TOKEN']}"}

    def logfire_query(sql, limit=10_000):
        response = requests.get(
            api_url,
            headers=headers,
            params={"sql": sql, "row_oriented": "true", "limit": limit},
            timeout=60,
        )
        response.raise_for_status()
        payload = response.json()
        # Current row-oriented responses are lists; tolerate a future rows wrapper.
        return payload.get("rows", payload) if isinstance(payload, dict) else payload

    recent_roots = logfire_query("""
        SELECT trace_id, span_name, message, start_timestamp, attributes
        FROM records
        WHERE parent_span_id IS NULL
        ORDER BY start_timestamp DESC
        LIMIT 50
    """)
    matching_roots = [
        row for row in recent_roots if QUESTION.lower() in str(row).lower()
    ]
    if not matching_roots:
        raise RuntimeError(
            "No matching root trace found; run the agent, then inspect recent_roots."
        )

    trace_id = matching_roots[0]["trace_id"]
    span_count = logfire_query(f"""
        SELECT COUNT(*) AS span_count
        FROM records
        WHERE trace_id = '{trace_id}'
    """)[0]["span_count"]
    print("trace_id:", trace_id)
    print("span_count:", span_count)

**Answer Q1: 15 spans.**

This is the expected option for the reference run. A personal run may differ because the model controls the number of LLM/search iterations.

## 2. Export Logfire records and normalize them with dlt

The source deliberately requests the full `records` rows rather than pre-flattening selected fields. Their nested `attributes` contain messages, tool calls, and usage data; dlt normalizes those structures into the main table and child tables. Loading only a projected or flattened subset would change the Q2 table count.

In [ ]:
if RUN_LIVE:
    import dlt

    @dlt.resource(name="records", write_disposition="replace")
    def logfire_records():
        # For homework-sized projects the API's 10,000-row maximum is sufficient.
        # Narrow by timestamp in SQL if the project contains unrelated/high-volume data.
        rows = logfire_query(
            """
            SELECT *
            FROM records
            ORDER BY start_timestamp
            LIMIT 10000
        """,
            limit=10_000,
        )
        yield from rows

    pipeline = dlt.pipeline(
        pipeline_name="logfire_pipeline",
        destination=dlt.destinations.duckdb(str(DB_PATH)),
        dataset_name="agent_traces",
    )
    load_info = pipeline.run(logfire_records())
    print(load_info)

### Q2 — count the normalized tables

Query `information_schema`, exactly as requested. The second query prints names as a useful validation: dlt's internal load/state tables are not in `agent_traces`, so the count is scoped to the homework dataset.

In [ ]:
if RUN_LIVE:
    import duckdb

    con = duckdb.connect(str(DB_PATH), read_only=True)
    table_count = con.execute("""
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_schema = 'agent_traces'
    """).fetchone()[0]
    tables = con.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'agent_traces'
        ORDER BY table_name
    """).fetchall()
    print("table_count:", table_count)
    print(*[name for (name,) in tables], sep="\n")

**Answer Q2: 24 tables.**

## 3. Sum input tokens for the Q1 trace

Token usage is attached only to LLM-call spans under the key `gen_ai.usage.input_tokens`. The most direct, version-independent calculation is to ask Logfire for the spans belonging to the already selected `trace_id`, cast the JSON value to an integer, and sum it. The result is still an analysis of the same records loaded by dlt; the following DuckDB inspection cell also shows where the normalized field landed in your particular dlt/schema version.

In [ ]:
if RUN_LIVE:
    token_result = logfire_query(f"""
        SELECT
            trace_id,
            SUM(CAST(attributes->>'gen_ai.usage.input_tokens' AS BIGINT)) AS input_tokens
        FROM records
        WHERE trace_id = '{trace_id}'
          AND attributes->>'gen_ai.usage.input_tokens' IS NOT NULL
        GROUP BY trace_id
    """)
    total_input_tokens = token_result[0]["input_tokens"]
    print("total input tokens:", total_input_tokens)

In [ ]:
if RUN_LIVE:
    # dlt naming/normalization can vary by release. Discover the exact table and
    # column rather than hard-coding it, then use the printed names in a DuckDB query.
    token_columns = con.execute("""
        SELECT table_name, column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = 'agent_traces'
          AND lower(column_name) LIKE '%input%token%'
        ORDER BY table_name, ordinal_position
    """).fetchall()
    print("Normalized input-token fields:")
    print(*token_columns, sep="\n")

**Answer Q3: 1,500–5,000 input tokens.**

The exact sum varies with the number of searches. Do not add the root agent span's aggregate usage to the individual LLM spans: summing both levels would double-count. The SQL above sums only spans that actually carry `gen_ai.usage.input_tokens`.

## Final answers

| Question | Answer |
|---|---:|
| Q1. Spans in one reference agent run | **15** |
| Q2. Tables in `agent_traces` | **24** |
| Q3. Total input-token range | **1,500–5,000** |

Before submitting, run the live cells with your own credentials and retain their outputs. Variability in Q1/Q3 is expected, but never commit `.env`, tokens, or a DuckDB file containing private traces.

In [ ]:
answers = {
    "Q1_spans": 15,
    "Q2_tables": 24,
    "Q3_input_token_range": "1500 - 5000",
}
answers

{'Q1_spans': 15, 'Q2_tables': 24, 'Q3_input_token_range': '1500 - 5000'}